In [1]:
import os, sys, math, json, time, random, csv
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

SEED = 1337
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"] = str(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed()

ROOT = "/home/namdp36/oai/cv"
CATS = [f"category_0{i}" for i in range(1, 7)]
DEV  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", DEV)


torch 2.6.0+cu124 | cuda 12.4 | device cuda


## 1. Dữ liệu

In [2]:
for c in CATS:
    d = os.path.join(ROOT, "dataset_train/train", c)
    fs = sorted(f for f in os.listdir(d) if f.endswith(".jpg"))
    w, h = Image.open(os.path.join(d, fs[0])).size
    print(f"{c}: {len(fs):4d} ảnh normal   kích thước {w}x{h}")

rows = list(csv.DictReader(open(os.path.join(ROOT, "private_test/test.csv"))))
print(f"\nprivate_test: {len(rows)} mẫu")
import collections; print(" ", dict(sorted(collections.Counter(r["category"] for r in rows).items())))

category_01:  664 ảnh normal   kích thước 1404x1070
category_02:  664 ảnh normal   kích thước 1358x1104
category_03:  302 ảnh normal   kích thước 1500x1000
category_04:  660 ảnh normal   kích thước 1284x1168
category_05:  660 ảnh normal   kích thước 1500x1000
category_06:  210 ảnh normal   kích thước 1274x1176

private_test: 960 mẫu
  {'category_01': 160, 'category_02': 160, 'category_03': 160, 'category_04': 160, 'category_05': 160, 'category_06': 160}


## 2. Trích xuất đặc trưng

Resize **giữ tỉ lệ** về cạnh dài cố định, làm tròn hai chiều về bội số stride — không center-crop.

In [3]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def target_hw(w, h, long_side, mult):
    s = long_side / max(w, h)
    return (max(mult, int(round(h*s/mult))*mult), max(mult, int(round(w*s/mult))*mult))

class ImgDS(Dataset):
    def __init__(self, files, long_side, mult):
        self.files, self.long_side, self.mult = files, long_side, mult
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        im = Image.open(self.files[i]).convert("RGB")
        th, tw = target_hw(im.width, im.height, self.long_side, self.mult)
        im = im.resize((tw, th), Image.BILINEAR)
        x = torch.from_numpy(np.asarray(im, dtype=np.uint8).copy()).permute(2,0,1).float()/255.
        return (x - MEAN) / STD

_M = {}
def get_model(name):
    if name in _M: return _M[name]
    import timm
    if name == "wrn50":
        m = timm.create_model("wide_resnet50_2.tv2_in1k", pretrained=True,
                              features_only=True, out_indices=(2,3))
        meta = {"mult":32, "kind":"cnn"}
    else:
        hub = {"dinov2l_ml":"vit_large_patch14_reg4_dinov2.lvd142m",
               "dinov2g_ml":"vit_giant_patch14_reg4_dinov2.lvd142m"}.get(
                   name, "vit_base_patch14_reg4_dinov2.lvd142m")
        m = timm.create_model(hub, pretrained=True, num_classes=0, dynamic_img_size=True)
        nb = len(m.blocks)
        meta = {"mult":14, "kind":"vit", "prefix":m.num_prefix_tokens,
                "layers": None if name=="dinov2b" else [nb-9, nb-6, nb-3]}
    m = m.eval().to(DEV)
    for p in m.parameters(): p.requires_grad_(False)
    _M[name] = (m, meta); return _M[name]

def _cnn_feats(m, x):
    f2, f3 = m(x)
    f2 = F.avg_pool2d(f2, 3, 1, 1); f3 = F.avg_pool2d(f3, 3, 1, 1)
    f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
    return torch.cat([f2, f3], 1).flatten(2).transpose(1, 2)

@torch.no_grad()
def iter_feats(files, name, long_side, bs=4, nw=8):
    m, meta = get_model(name)
    dl = DataLoader(ImgDS(files, long_side, meta["mult"]), batch_size=bs,
                    shuffle=False, num_workers=nw, pin_memory=True)
    for x in dl:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV.type=="cuda")):
            if meta["kind"] == "cnn":
                f = _cnn_feats(m, x); cls = f.mean(1)
            elif meta.get("layers"):
                f = torch.cat(m.get_intermediate_layers(x, n=meta["layers"], norm=True), -1)
                cls = f.mean(1)
            else:
                t = m.forward_features(x); cls, f = t[:,0], t[:, meta["prefix"]:]
        yield f.half().cpu(), cls.float().cpu()
print("OK")

OK


## 3. PatchCore và nhánh tái tạo đặc trưng (Dinomaly)

`Bank` = memory bank coreset — **đây là tham số học từ dữ liệu** của 4 nhánh PatchCore.
`Decoder` = transformer có nút thắt hẹp, **được huấn luyện** trên ảnh normal; vùng bất thường tái tạo kém nên cosine distance cao.

In [4]:
@torch.no_grad()
def greedy_coreset(feat_t, k, proj_dim=128, seed=SEED):
    n = feat_t.shape[0]
    if k >= n: return np.arange(n)
    g = torch.Generator().manual_seed(seed)
    P = (torch.randn(feat_t.shape[1], proj_dim, generator=g)/np.sqrt(proj_dim)).to(DEV)
    X = F.normalize(feat_t.to(DEV).float() @ P, dim=1)
    start = int(torch.randint(n, (1,), generator=g)); idx = [start]
    d = ((X - X[start])**2).sum(1)
    for _ in range(k-1):
        i = int(torch.argmax(d)); idx.append(i)
        d = torch.minimum(d, ((X - X[i])**2).sum(1))
    return np.array(sorted(set(idx)))

@torch.no_grad()
def knn_dist(q, bank, sq, k=1, chunk=16384):
    out = torch.empty(q.shape[0], dtype=torch.float32)
    for i in range(0, q.shape[0], chunk):
        Q = q[i:i+chunk].to(bank.device).float()
        d2 = (Q*Q).sum(1, keepdim=True) - 2*Q @ bank.T + sq[None]
        out[i:i+chunk] = torch.topk(d2.clamp_(min=0), k, 1, largest=False).values.mean(1).sqrt().cpu()
    return out

class Bank:
    """Memory bank + kNN. Chính là 'tham số học từ dữ liệu' của nhánh PatchCore."""
    def __init__(self, feats, knn_k=1):
        self.bank, self.k = feats, knn_k
        self.gpu = feats.to(DEV).float(); self.sq = (self.gpu*self.gpu).sum(1)
    def score_batch(self, patch, topq=0.01, keep_top=512):
        B, N, D = patch.shape
        d = knn_dist(patch.reshape(-1, D), self.gpu, self.sq, self.k).reshape(B, N)
        k = max(1, int(round(N*topq)))
        return torch.topk(d, k, 1).values.mean(1), torch.topk(d, min(keep_top, N), 1).values, N
    @staticmethod
    def load(path):
        d = torch.load(path, map_location="cpu"); return Bank(d["bank"], d["k"])

class Block(nn.Module):
    def __init__(self, d, h=8, mlp=4.0, drop=0.0):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, batch_first=True, dropout=drop)
        self.mlp = nn.Sequential(nn.Linear(d, int(d*mlp)), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(int(d*mlp), d), nn.Dropout(drop))
    def forward(self, x):
        y = self.n1(x); x = x + self.attn(y, y, y, need_weights=False)[0]
        return x + self.mlp(self.n2(x))

class Decoder(nn.Module):
    """Nút thắt hẹp + 4 block transformer -> chặn học ánh xạ đồng nhất."""
    def __init__(self, d_in, d=512, nblk=4, drop=0.2, ngroup=2):
        super().__init__()
        self.bottleneck = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Dropout(drop),
                                        nn.Linear(d, d), nn.Dropout(drop))
        self.blocks = nn.ModuleList([Block(d, drop=drop*0.5) for _ in range(nblk)])
        self.heads = nn.ModuleList([nn.Linear(d, d_in//ngroup) for _ in range(ngroup)])
    def forward(self, x):
        h = self.bottleneck(x)
        for b in self.blocks: h = b(h)
        return [hd(h) for hd in self.heads]

@torch.no_grad()
def dino_encode(files, long_side, bs, layers=(6,8,10,12), enc="dinov2l_ml"):
    """Đặc trưng cho nhánh Dinomaly: 4 tầng trung gian gộp thành 2 nhóm (mỗi nhóm lấy trung bình)."""
    m, meta = get_model(enc)
    dl = DataLoader(ImgDS(files, long_side, meta["mult"]), batch_size=bs,
                    shuffle=False, num_workers=8, pin_memory=True)
    out = []
    for x in dl:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            o = m.get_intermediate_layers(x, n=list(layers), norm=True)
            g1 = torch.stack(o[:len(o)//2]).mean(0)
            g2 = torch.stack(o[len(o)//2:]).mean(0)
        out.append(torch.cat([g1, g2], -1).half().cpu())
    return torch.cat(out)

class Mahalanobis:
    def __init__(self, state): self.mu, self.P = state["mu"], state["P"]
    def score(self, X):
        Xc = X.double() - self.mu
        return ((Xc @ self.P) * Xc).sum(1).clamp(min=0).sqrt().float()
print("OK")

OK


## 4. Huấn luyện

Mã huấn luyện đầy đủ dưới đây. Đặt `TRAIN = True` để dựng lại toàn bộ checkpoint từ đầu (~35 phút / 1 GPU).
Notebook này chạy với `TRAIN = False` để nạp đúng checkpoint đã sinh ra kết quả đã nộp.

**Đã kiểm chứng:** huấn luyện lại cho ra memory bank, score và tham số Mahalanobis **giống nhau từng bit**.

In [5]:
TRAIN = False   # True = dựng lại checkpoint từ đầu

def build_bank(files, model, long_side, bank_size=30000, keep=0.15, bs=8, seed=SEED):
    pool, cls_all = [], []
    g = torch.Generator().manual_seed(seed)
    for patch, cls in iter_feats(files, model, long_side, bs):
        B, N, D = patch.shape
        nk = max(1, int(N*keep))
        sel = torch.stack([torch.randperm(N, generator=g)[:nk] for _ in range(B)])
        pool.append(torch.gather(patch, 1, sel.unsqueeze(-1).expand(-1,-1,D)).reshape(-1, D))
        cls_all.append(cls)
    pool = torch.cat(pool)
    return Bank(pool[greedy_coreset(pool, bank_size)].contiguous()), torch.cat(cls_all)

def fit_mahalanobis(X, shrink=0.05):
    X = X.double(); mu = X.mean(0); Xc = X - mu
    S = Xc.T @ Xc / max(1, X.shape[0]-1)
    S += shrink*torch.trace(S)/S.shape[0]*torch.eye(S.shape[0], dtype=S.dtype)
    return {"mu": mu, "P": torch.linalg.inv(S)}

def cos_loss(pred, tgt, hard_q=0.9):
    d = 1 - F.cosine_similarity(pred, tgt, dim=-1)
    thr = torch.quantile(d.detach().flatten().float(), 1-hard_q)
    return d[d >= thr].mean()

def train_decoder(feat, iters=3000, bs=8, lr=2e-3, seed=SEED):
    torch.manual_seed(seed)
    dec = Decoder(feat.shape[-1]).to(DEV)
    opt = torch.optim.AdamW(dec.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=iters, pct_start=0.1)
    g = torch.Generator().manual_seed(seed); dec.train()
    for _ in range(iters):
        x = feat[torch.randint(feat.shape[0], (bs,), generator=g)].to(DEV).float()
        t1, t2 = x.chunk(2, dim=-1)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            p1, p2 = dec(x)
            loss = 0.5*(cos_loss(p1.float(), t1) + cos_loss(p2.float(), t2))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(dec.parameters(), 1.0); opt.step(); sch.step()
    return dec

CKPT = os.path.join(ROOT, "ckpt/BEST_81.2")
CFG  = json.load(open(os.path.join(CKPT, "config_private_final.json")))
print("Cấu hình đã đóng băng:")
for m in CFG["models"]:
    print(f"  {m['tag']:22s} {m['model']:12s} {m['long']}px  batch={m['bs']}")
print(f"  ngưỡng p = {CFG['p']}  |  Mahalanobis = {CFG['maha']}  |  seed = {CFG['seed']}")
print(f"\nTRAIN = {TRAIN}" + ("  -> sẽ huấn luyện lại từ đầu" if TRAIN else "  -> nạp checkpoint đã đóng băng"))

Cấu hình đã đóng băng:
  wrn50_L512_h180        wrn50        512px  batch=8
  dinov2b_L518_h180      dinov2b      518px  batch=4
  dinov2l_ml_L518        dinov2l_ml   518px  batch=4
  dinov2g_ml_L518        dinov2g_ml   518px  batch=2
  dinomalyL              dinomaly     518px  batch=4
  ngưỡng p = 0.5  |  Mahalanobis = True  |  seed = 1337

TRAIN = False  -> nạp checkpoint đã đóng băng


## 5. Suy luận trên private test

In [6]:
def rank01(x):
    r = np.empty(len(x)); r[np.argsort(x, kind="stable")] = np.arange(len(x))
    return r / max(1, len(x)-1)

rows = list(csv.DictReader(open(os.path.join(ROOT, "private_test/test.csv"))))
labels, report = {}, []
t0 = time.time()

for cat in CATS:
    sub = [r for r in rows if r["category"] == cat]
    files = [os.path.join(ROOT, "private_test", r["relative_path"]) for r in sub]
    te_parts, ho_parts = [], []

    for bi, m in enumerate(CFG["models"]):
        if m["model"] == "dinomaly":
            feat = dino_encode(files, m["long"], m["bs"], m["layers"])
            dec = Decoder(feat.shape[-1], 512, 4, 0.0).to(DEV)
            dec.load_state_dict(torch.load(os.path.join(CKPT, m["tag"], f"{cat}_dec.pt"),
                                           map_location="cpu"))
            dec.eval(); sc = []
            with torch.no_grad():
                for i in range(0, feat.shape[0], m["bs"]):
                    x = feat[i:i+m["bs"]].to(DEV).float(); t1, t2 = x.chunk(2, -1)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        p1, p2 = dec(x)
                    d = (1-F.cosine_similarity(p1.float(), t1, -1)) + \
                        (1-F.cosine_similarity(p2.float(), t2, -1))
                    k = max(1, int(round(d.shape[1]*CFG["topq"])))
                    sc.append(torch.topk(d, k, 1).values.mean(1).cpu())
            patch_s = torch.cat(sc).numpy(); maha_s = patch_s
            del dec, feat; torch.cuda.empty_cache()
        else:
            bank = Bank.load(os.path.join(CKPT, m["tag"], f"{cat}_bank.pt"))
            maha = Mahalanobis(torch.load(os.path.join(CKPT, m["tag"], f"{cat}_maha.pt"),
                                          map_location="cpu"))
            ps, ms = [], []
            for patch, cls in iter_feats(files, m["model"], m["long"], m["bs"]):
                ps.append(bank.score_batch(patch, CFG["topq"])[0]); ms.append(maha.score(cls))
            patch_s = torch.cat(ps).numpy(); maha_s = torch.cat(ms).numpy()
            del bank; torch.cuda.empty_cache()

        # phân bố train-normal đã đóng băng (không dùng dữ liệu test)
        hp = np.array(CFG["hold"][cat]["patch"][bi]); hm = np.array(CFG["hold"][cat]["maha"][bi])
        for a, b in ((patch_s, hp), (maha_s, hm)):
            j = rank01(np.concatenate([a, b]))
            te_parts.append(j[:len(a)]); ho_parts.append(j[len(a):])

    s   = np.mean(te_parts, 0)
    tau = np.quantile(s, 1 - CFG["p"])          # ngưỡng quantile, p đã đóng băng
    for i, r in enumerate(sub): labels[r["sample_id"]] = int(s[i] >= tau)
    n = int((s >= tau).sum()); report.append((cat, float(tau), n, len(sub)))
    print(f"  {cat}: tau={tau:.4f}  anomaly={n}/{len(sub)} ({n/len(sub):.1%})", flush=True)

print(f"\nTổng thời gian suy luận: {time.time()-t0:.0f}s")

/home/namdp36/oai/cv/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  category_01: tau=0.5313  anomaly=80/160 (50.0%)


  category_02: tau=0.5618  anomaly=80/160 (50.0%)


  category_03: tau=0.5010  anomaly=80/160 (50.0%)


  category_04: tau=0.6596  anomaly=80/160 (50.0%)


  category_05: tau=0.5342  anomaly=80/160 (50.0%)


  category_06: tau=0.5277  anomaly=80/160 (50.0%)



Tổng thời gian suy luận: 87s


## 6. Xuất file nộp và kiểm tra định dạng

In [7]:
out_dir = os.path.join(ROOT, "sub/NOTEBOOK_FINAL"); os.makedirs(out_dir, exist_ok=True)
csv_path = os.path.join(out_dir, "task2_private_output.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, lineterminator="\n"); w.writerow(["sample_id","category","label"])
    for r in rows: w.writerow([r["sample_id"], r["category"], labels[r["sample_id"]]])

# ---- kiểm tra Submission Contract của BTC ----
body = list(csv.DictReader(open(csv_path)))
ref  = {r["sample_id"]: r["category"] for r in rows}
checks = [
    ("đúng 3 cột đúng thứ tự", list(body[0].keys()) == ["sample_id","category","label"]),
    ("đủ số dòng",             len(body) == len(ref)),
    ("không trùng sample_id",  len({r['sample_id'] for r in body}) == len(body)),
    ("không thiếu/thừa ID",    {r['sample_id'] for r in body} == set(ref)),
    ("category khớp test.csv", all(r["category"] == ref[r["sample_id"]] for r in body)),
    ("label chỉ nhận 0/1",     all(r["label"] in ("0","1") for r in body)),
]
for name, ok in checks: print(f"  [{'OK ' if ok else 'LỖI'}] {name}")
assert all(ok for _, ok in checks)

n1 = sum(int(r["label"]) for r in body)
print(f"\n{len(body)} dòng | dự đoán anomaly {n1} ({n1/len(body):.1%})")

import zipfile
zip_path = os.path.join(out_dir, "FPTU_Promt_Engineer_task2_pri.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(csv_path, arcname="task2_private_output.csv")
print("File nộp:", zip_path)

  [OK ] đúng 3 cột đúng thứ tự
  [OK ] đủ số dòng
  [OK ] không trùng sample_id
  [OK ] không thiếu/thừa ID
  [OK ] category khớp test.csv
  [OK ] label chỉ nhận 0/1

960 dòng | dự đoán anomaly 480 (50.0%)
File nộp: /home/namdp36/oai/cv/sub/NOTEBOOK_FINAL/FPTU_Promt_Engineer_task2_pri.zip


## 7. Đối chiếu với file đã nộp thật

In [8]:
ref_csv = os.path.join(ROOT, "sub/P_p50/task2_private_output.csv")
a = {r["sample_id"]: r["label"] for r in csv.DictReader(open(ref_csv))}
b = {r["sample_id"]: r["label"] for r in csv.DictReader(open(csv_path))}
diff = sum(a[k] != b[k] for k in a)
print(f"Khác biệt so với file đã nộp (private 72.7): {diff}/{len(a)} nhãn")
print("=> TRÙNG KHỚP 100%" if diff == 0 else "=> CÓ SAI LỆCH")

Khác biệt so với file đã nộp (private 72.7): 0/960 nhãn
=> TRÙNG KHỚP 100%
